# Student Dropout Classification — CSE 6023

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hhsonet/ML/blob/main/notebooks/student-dropout-classification.ipynb)

**Problem:** classify student records as `Complete` or `Dropout`.  
**Dataset:** `data/ml_dp_l_e_v_4_2.csv` (4,624 rows; encoded academic/demographic features).  
**Goal:** compare several classical machine-learning classifiers using the same train/test split and clear evaluation metrics.

> Privacy note: the source dataset contains student-related demographic, academic, and financial attributes. Confirm that public sharing is authorized and sufficiently de-identified before using it outside coursework.

## 1. Load the data

The notebook reads the dataset from this GitHub repository so it works in Colab without a Google Drive mount.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

DATA_URL = "https://raw.githubusercontent.com/hhsonet/ML/main/data/ml_dp_l_e_v_4_2.csv"
data = pd.read_csv(DATA_URL)

print("Dataset shape:", data.shape)
data.head()

## 2. Explore the data (EDA)

Check the target distribution, selected feature types, and missing values before training.

In [ ]:
FEATURES = [
    "Dept", "Gender", "District", "Age", "SSC_Board", "HSC_Board",
    "SSC_RES", "HSC",
    "Course_1", "Course_2", "Course_3", "Course_4", "Course_5",
    "Course_6", "Course_7", "Course_8", "Course_9", "Course_10",
]
TARGET = "Status"

print(data[TARGET].value_counts())
print("\nMissing values in selected features:")
print(data[FEATURES].isna().sum().sort_values(ascending=False).head(10))

data[TARGET].value_counts().plot(kind="bar", title="Target class distribution")
plt.xlabel("Status")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

## 3. Preprocess the data

Numeric coercion plus median imputation makes the workflow robust to malformed or missing numeric values. The target is label-encoded. Scaling is used only for algorithms that benefit from it.

In [ ]:
X = data[FEATURES].apply(pd.to_numeric, errors="coerce")
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data[TARGET])

print("Target classes:", list(label_encoder.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

def scaled_model(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

def tree_model(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])

## 4. Train the models

The same holdout split is used for every model so the comparison is consistent.

In [ ]:
models = {
    "SVM": scaled_model(SVC(C=25, gamma=0.001, probability=True, random_state=42)),
    "Gaussian NB": scaled_model(GaussianNB()),
    "KNN": scaled_model(KNeighborsClassifier(n_neighbors=16)),
    "Decision Tree": tree_model(DecisionTreeClassifier(max_depth=6, random_state=42)),
    "Logistic Regression": scaled_model(LogisticRegression(max_iter=2000, random_state=42)),
    "AdaBoost": tree_model(AdaBoostClassifier(n_estimators=100, random_state=42)),
    "Random Forest": tree_model(RandomForestClassifier(n_estimators=200, random_state=42)),
}

fitted_models = {}
rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, proba),
    })

results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
results

## 5. Evaluate the models

Show the best model's detailed classification report and confusion matrix. The comparison table above is the main result summary.

In [ ]:
best_name = results.loc[0, "Model"]
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)

print("Best model by F1:", best_name)
print(classification_report(y_test, best_pred, target_names=label_encoder.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_pred,
    display_labels=label_encoder.classes_,
    cmap="Blues",
)
plt.title(f"Confusion Matrix — {best_name}")
plt.show()

## 6. Conclusion

This cleaned workflow avoids the original cumulative-metrics bug by calculating each model's metrics independently. Use the generated comparison table to discuss the trade-off between accuracy, F1 score, ROC-AUC, and model complexity.

For a stronger academic evaluation, add repeated cross-validation and report mean ± standard deviation.